# Adım 1: EPİAŞ'tan veri çekme (PTF, üretim, tüketim)

Bu not defteri, ilk analiz için gereken saatlik verileri EPİAŞ Şeffaflık Platformu API'sinden çeker, kontrol eder ve tek bir ana tabloda birleştirir.

**Çekilen veriler (1 Ocak 2023 - 31 Ekim 2026, saatlik):**
1. Piyasa Takas Fiyatı (PTF)
2. Gerçek zamanlı üretim (kaynak bazında)
3. Gerçek zamanlı tüketim
4. Opsiyonel: SMF ve sistem yönü (ikinci analiz için)

**Güvenlik notları**
- Kullanıcı adı ve şifre koda yazılmaz. Çalıştırınca sorulur (`getpass`), ekrana basılmaz ve dosyaya kaydedilmez.
- Sadece kendi kişisel hesabınızla çalıştırın.
- İndirilen ham veri `01_ham_veri/` klasörüne aylık dosyalar olarak kaydedilir. Tekrar çalıştırınca mevcut aylar atlanır.

**Klasör yapısı (bu dosyanın bulunduğu yerde oluşturulur)**
```
01_ham_veri/     API'den inen aylık ham dosyalar
02_islenmis/     birleştirilmiş ana tablo
```


In [ ]:
# 1) Kurulum (bir kez çalıştırın)
# !pip install requests pandas pyarrow matplotlib

In [ ]:
# 2) Ayarlar
import os, time, json, getpass
from datetime import date, timedelta
from pathlib import Path
import requests
import pandas as pd

# ADIM 1: İlk çalıştırmada DENEME_MODU = True bırakın (sadece Ekim 2026 çekilir).
# Her şey yolunda giderse False yapıp tüm aralığı çekin.
DENEME_MODU = True

if DENEME_MODU:
    BASLANGIC, BITIS = date(2026, 10, 1), date(2026, 10, 31)
else:
    BASLANGIC, BITIS = date(2023, 1, 1), date(2026, 10, 31)

# Opsiyonel veri setleri (şimdilik kapalı, ikinci analiz için True yapın)
SMF_VE_SISTEM_YONU_CEK = False

HAM  = Path("01_ham_veri");  HAM.mkdir(exist_ok=True)
ISLN = Path("02_islenmis");  ISLN.mkdir(exist_ok=True)

LOGIN_URL = "https://giris.epias.com.tr/cas/v1/tickets"
API_ROOT  = "https://seffaflik.epias.com.tr/electricity-service"

# Uç noktalar (EPİAŞ Şeffaflık API v1, POST isteği)
ENDPOINTS = {
    "ptf":          "/v1/markets/dam/data/mcp",
    "uretim":       "/v1/generation/data/realtime-generation",
    "tuketim":      "/v1/consumption/data/realtime-consumption",
    "smf":          "/v1/markets/bpm/data/system-marginal-price",
    "sistem_yonu":  "/v1/markets/bpm/data/system-direction",
}
SECILENLER = ["ptf", "uretim", "tuketim"] + (["smf", "sistem_yonu"] if SMF_VE_SISTEM_YONU_CEK else [])
print("Çekilecek veri setleri:", SECILENLER, "| Aralık:", BASLANGIC, "-", BITIS)

In [ ]:
# 3) Giriş: TGT (Ticket Granting Ticket) alma
# Kullanıcı adınız kayıt e-postanızdır. Bilgiler ekrana yazılmaz, kaydedilmez.
class Oturum:
    def __init__(self):
        self.kullanici = os.environ.get("EPIAS_USER") or input("EPİAŞ kullanıcı adı (e-posta): ").strip()
        self._sifre    = os.environ.get("EPIAS_PASS") or getpass.getpass("EPİAŞ şifre: ")
        self.tgt = None
        self.tgt_zamani = 0

    def yenile(self):
        r = requests.post(
            LOGIN_URL,
            data={"username": self.kullanici, "password": self._sifre},
            headers={"Content-Type": "application/x-www-form-urlencoded", "Accept": "text/plain"},
            timeout=30,
        )
        if r.status_code not in (200, 201) or not r.text.strip().startswith("TGT-"):
            raise RuntimeError(f"Giriş başarısız (HTTP {r.status_code}). Kullanıcı adı/şifreyi kontrol edin.")
        self.tgt = r.text.strip()
        self.tgt_zamani = time.time()

    def baslik(self):
        # Bilet birkaç saat geçerlidir; 2 saatte bir yenilemek güvenli tarafta kalır
        if self.tgt is None or time.time() - self.tgt_zamani > 2 * 3600:
            self.yenile()
        return {"TGT": self.tgt, "Content-Type": "application/json", "Accept": "application/json"}

oturum = Oturum()
oturum.yenile()
print("Giriş başarılı.")

In [ ]:
# 4) Aylık parçalarla veri çekme (uzun aralıklarda hata riskini azaltır) ve önbellekleme
def ay_araliklari(bas, bit):
    d = date(bas.year, bas.month, 1)
    while d <= bit:
        sonraki = date(d.year + (d.month // 12), (d.month % 12) + 1, 1)
        ay_son = min(sonraki - timedelta(days=1), bit)
        ay_bas = max(d, bas)
        yield ay_bas, ay_son
        d = sonraki

def iso(d):
    return f"{d.isoformat()}T00:00:00+03:00"

def api_cagri(yol, bas, bit, deneme=4):
    url = API_ROOT + yol
    govde = {"startDate": iso(bas), "endDate": iso(bit)}
    for i in range(deneme):
        try:
            r = requests.post(url, headers=oturum.baslik(), json=govde, timeout=90)
            if r.status_code in (401, 403):
                oturum.yenile(); continue
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(2 ** (i + 1)); continue
            r.raise_for_status()
            return r.json()
        except requests.exceptions.RequestException:
            if i == deneme - 1: raise
            time.sleep(2 ** (i + 1))
    raise RuntimeError(f"Çağrı başarısız: {yol} {bas}-{bit}")

def veri_seti_cek(ad):
    parcalar = []
    for bas, bit in ay_araliklari(BASLANGIC, BITIS):
        dosya = HAM / f"{ad}_{bas:%Y-%m}.json"
        if dosya.exists():
            veri = json.loads(dosya.read_text(encoding="utf-8"))
        else:
            veri = api_cagri(ENDPOINTS[ad], bas, bit)
            dosya.write_text(json.dumps(veri, ensure_ascii=False), encoding="utf-8")
            time.sleep(0.5)   # platforma nazik davranalım
        kayitlar = veri.get("items", [])
        if kayitlar:
            df = pd.DataFrame(kayitlar); df["_ay"] = f"{bas:%Y-%m}"
            parcalar.append(df)
        print(f"{ad} {bas:%Y-%m}: {len(kayitlar)} kayıt")
    return pd.concat(parcalar, ignore_index=True) if parcalar else pd.DataFrame()

ham = {ad: veri_seti_cek(ad) for ad in SECILENLER}

In [ ]:
# 5) Ham tabloyu inceleyin: sütun adlarını görmeden devam etmeyin
for ad, df in ham.items():
    print("="*60); print(ad, df.shape); print(list(df.columns)); display(df.head(3))

In [ ]:
# 6) Zaman damgasını standartlaştır ve saatlik tabloya çevir
def zaman_damgasi(df):
    d = pd.to_datetime(df["date"], utc=True).dt.tz_convert("Europe/Istanbul")
    # Bazı servislerde 'date' sadece günü, 'hour' ya da 'time' saati verir
    for kol in ("hour", "time"):
        if kol in df.columns and d.dt.hour.nunique() == 1:
            saat = pd.to_numeric(df[kol].astype(str).str.slice(0, 2), errors="coerce").fillna(0)
            d = d + pd.to_timedelta(saat, unit="h")
            break
    return d.dt.tz_localize(None)   # Türkiye UTC+3 sabit, yaz saati yok; saat dilimi bilgisini kaldırıyoruz

def hazirla(ad, df, deger_kolonlari):
    if df.empty: return pd.DataFrame()
    out = df.copy()
    out["zaman"] = zaman_damgasi(out)
    out = out.drop_duplicates("zaman").set_index("zaman").sort_index()
    return out[[k for k in deger_kolonlari if k in out.columns]]

ptf = hazirla("ptf", ham["ptf"], ["price", "priceUsd", "priceEur"]).rename(
    columns={"price": "ptf", "priceUsd": "ptf_usd", "priceEur": "ptf_eur"})
tuk = hazirla("tuketim", ham["tuketim"], ["consumption"]).rename(columns={"consumption": "tuketim_mwh"})

URETIM_KOLONLARI = ["total","naturalGas","dammedHydro","lignite","river","importCoal","wind","sun",
                    "fueloil","geothermal","asphaltiteCoal","blackCoal","biomass","naphta","lng","importExport","wasteheat"]
uret = hazirla("uretim", ham["uretim"], URETIM_KOLONLARI).add_prefix("uretim_")
uret = uret.rename(columns={"uretim_sun": "uretim_solar"})   # API güneşi "sun" adıyla verir

ana = ptf.join([tuk, uret], how="outer")

if SMF_VE_SISTEM_YONU_CEK:
    smf = hazirla("smf", ham["smf"], ["price"]).rename(columns={"price": "smf"})
    sy  = hazirla("sistem_yonu", ham["sistem_yonu"], ["systemDirection", "direction"])
    sy.columns = ["sistem_yonu"] * len(sy.columns) if len(sy.columns) == 1 else sy.columns
    ana = ana.join([smf, sy], how="outer")

ana.index.name = "zaman"
print(ana.shape); display(ana.head()); display(ana.tail())

In [ ]:
# 7) Kalite kontrol (yayınlamadan önce mutlaka okuyun)
beklenen = pd.date_range(f"{BASLANGIC} 00:00", f"{BITIS} 23:00", freq="h")
eksik_saat = beklenen.difference(ana.index)
print(f"Beklenen saat: {len(beklenen)}  |  Tabloda: {len(ana)}  |  Eksik: {len(eksik_saat)}")
if len(eksik_saat): print("İlk eksik saatler:", list(eksik_saat[:10]))

print("\nSütun başına boş değer oranı (%):")
print((ana.reindex(beklenen).isna().mean() * 100).round(2).to_string())

print("\nPTF özet istatistik:"); print(ana["ptf"].describe().round(1).to_string())
print("\nPTF 0 veya negatif saat sayısı:", int((ana["ptf"] <= 0).sum()))

# Üretim toplamı ile kaynak toplamı tutarlı mı?
kaynaklar = [c for c in ana.columns if c.startswith("uretim_") and c != "uretim_total"]  # importExport toplama dahildir
if "uretim_total" in ana.columns and kaynaklar:
    fark = (ana["uretim_total"] - ana[kaynaklar].sum(axis=1)).abs()
    print("\nÜretim toplamı ile kaynak toplamı farkı (MWh) medyan/maks:", round(fark.median(),1), round(fark.max(),1))

In [ ]:
# 8) Aylık özet: PTF ortalaması bir bakışta mantıklı mı?
ay = ana["ptf"].resample("MS").mean().round(0)
display(ay.to_frame("ptf_ort_TL_MWh").T if len(ay) <= 12 else ay.to_frame("ptf_ort_TL_MWh").tail(24))

import matplotlib.pyplot as plt
ana["ptf"].resample("MS").mean().plot(figsize=(10,3), title="Aylık ortalama PTF (TL/MWh)")
plt.show()

In [ ]:
# 9) Kaydet
ana.to_csv(ISLN / "ana_tablo_saatlik.csv", encoding="utf-8-sig")
ana.to_parquet(ISLN / "ana_tablo_saatlik.parquet")
print("Kaydedildi:", ISLN / "ana_tablo_saatlik.csv")
# Tam veriyle çalıştırdıysanız 02_hafta01_analiz.ipynb dosyasını açın.